In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
from PIL import Image

from lerobot.policies.factory import make_pre_post_processors

from vla.models.smolvla import smolvla
from vla.constants import PROJECT_ROOT

CHECKPOINT = "lerobot/smolvla_base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Offline data settings ---
# Suite name under data/images/libero/ (e.g. "object", "spatial")
SUITE = "object"
# Frame index to use from the episode
FRAME_IDX = 0


In [ ]:
def _find_episode_dirs(suite: str) -> list[tuple[Path, str]]:
    """Return list of (frames_dir, task_description) for a suite.

    The exported data layout has two sibling directories per episode:
      - ep<NNNN>_<task_slug>/task.txt   (contains the task description)
      - ep<NNNN>_task_<ID>/frame*.png   (contains the actual frames)
    This function pairs them by matching the ep<NNNN> prefix.
    """
    suite_dir = PROJECT_ROOT / "data" / "images" / "libero" / suite
    if not suite_dir.exists():
        raise FileNotFoundError(f"Suite directory not found: {suite_dir}")

    # Collect task descriptions and frame dirs separately, keyed by episode prefix
    task_descs: dict[str, str] = {}   # ep_prefix -> description
    frame_dirs: dict[str, Path] = {}  # ep_prefix -> directory with frames

    for child in sorted(suite_dir.iterdir()):
        if not child.is_dir():
            continue
        # Extract episode prefix, e.g. "ep0000"
        ep_prefix = child.name.split("_")[0]  # "ep0000"
        task_file = child / "task.txt"
        if task_file.exists():
            task_descs[ep_prefix] = task_file.read_text(encoding="utf-8").strip()
        if any(child.glob("frame*.png")):
            frame_dirs[ep_prefix] = child

    # Pair them up
    results = []
    for ep_prefix in sorted(frame_dirs):
        desc = task_descs.get(ep_prefix, "")
        results.append((frame_dirs[ep_prefix], desc))
    return results
print("Available episodes in suite:", _find_episode_dirs(SUITE))

In [ ]:
def _load_offline_observation(
    frames_dir: Path,
    frame_idx: int,
    task_description: str,
    state_dim: int,
    image_keys: list[str],
    image_size: int = 256,
    device: torch.device | None = None,
) -> tuple[dict, np.ndarray]:
    """Load a saved PNG frame and build a policy-ready batch dict.

    The single frame is duplicated across all expected camera keys so the
    policy receives the right number of image inputs.

    Returns (batch, img_np_rgb) where img_np_rgb is HxWx3 uint8 for visualization.
    """
    frame_path = frames_dir / f"frame{frame_idx:04d}.png"
    if not frame_path.exists():
        raise FileNotFoundError(f"Frame not found: {frame_path}")

    img_pil = Image.open(frame_path).convert("RGB")
    # Resize to match model expectation
    img_pil = img_pil.resize((image_size, image_size), Image.BILINEAR)
    img_np = np.array(img_pil)  # H, W, 3  uint8

    # Convert to tensor [1, C, H, W] float32 in [0, 1]
    img_tensor = torch.from_numpy(img_np).float() / 255.0
    img_tensor = img_tensor.permute(2, 0, 1).unsqueeze(0).contiguous()  # [1, 3, H, W]
    if device is not None:
        img_tensor = img_tensor.to(device, non_blocking=True)

    # Dummy robot state (zeros) — the attention analysis focuses on vision/language
    state = torch.zeros(1, state_dim, dtype=torch.float32)
    if device is not None:
        state = state.to(device, non_blocking=True)

    batch: dict = {
        "observation.state": state,
        "task": [task_description],
    }
    # Duplicate the single image into every expected camera slot
    for key in image_keys:
        batch[key] = img_tensor.clone()

    return batch, img_np


In [ ]:
def overlay_heatmap(img_np, visual_attention_1d):
    """
    Reshapes 64 visual tokens into an 8x8 grid and overlays it on the camera view.
    """
    # Normalize image to 0-255 uint8 if it's in float 0-1
    if img_np.max() <= 1.0:
        img_np = (img_np * 255).astype(np.uint8)
        
    # Reshape 64 tokens to 8x8
    heatmap_8x8 = visual_attention_1d.reshape(8, 8).numpy()
    
    # Normalize heatmap for visualization
    heatmap_norm = heatmap_8x8 - heatmap_8x8.min()
    heatmap_norm = heatmap_norm / (heatmap_norm.max() + 1e-8)
    
    # Resize to match original image dimensions
    heatmap_resized = cv2.resize(heatmap_norm, (img_np.shape[1], img_np.shape[0]))
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    
    # Blend image and heatmap
    overlay = cv2.addWeighted(img_np, 0.6, heatmap_colored, 0.4, 0)
    return overlay

In [ ]:
def main():
    device_obj = torch.device(DEVICE)
    policy, model_id, _ = smolvla(CHECKPOINT, str(device_obj))
    policy.eval()
    
    # Identify state dim
    state_feature = policy.config.input_features.get("observation.state")
    state_dim = state_feature.shape[0] if state_feature else 8

    preprocessor, postprocessor = make_pre_post_processors(
        policy.config,
        model_id,
        preprocessor_overrides={"device_processor": {"device": str(device_obj)}},
    )
    
    # 1. Monkey-patch eager_attention_forward to capture attention weights.
    #    SmolVLA bypasses standard LlamaAttention.forward — it manually computes
    #    Q/K/V and calls vlm_with_expert.eager_attention_forward which discards
    #    the attention probability matrix.  We wrap it to save a copy.

    vwe = policy.model.vlm_with_expert
    captured_attn_probs: list[torch.Tensor] = []
    _original_eager_attn = vwe.eager_attention_forward

    def _patched_eager_attention_forward(
        self, attention_mask, batch_size, head_dim, query_states, key_states, value_states
    ):
        num_att_heads = self.num_attention_heads
        num_key_value_heads = self.num_key_value_heads
        num_key_value_groups = num_att_heads // num_key_value_heads
        sequence_length = key_states.shape[1]

        key_states = key_states[:, :, :, None, :].expand(
            batch_size, sequence_length, num_key_value_heads, num_key_value_groups, head_dim
        ).reshape(batch_size, sequence_length, num_key_value_heads * num_key_value_groups, head_dim)

        value_states = value_states[:, :, :, None, :].expand(
            batch_size, sequence_length, num_key_value_heads, num_key_value_groups, head_dim
        ).reshape(batch_size, sequence_length, num_key_value_heads * num_key_value_groups, head_dim)

        query_states = query_states.to(dtype=torch.float32).transpose(1, 2)
        key_states_t = key_states.to(dtype=torch.float32).transpose(1, 2)

        att_weights = torch.matmul(query_states, key_states_t.transpose(2, 3)) * (head_dim ** -0.5)
        att_weights = att_weights.to(dtype=torch.float32)
        big_neg = torch.finfo(att_weights.dtype).min
        masked_att_weights = torch.where(attention_mask[:, None, :, :], att_weights, big_neg)
        probs = nn.functional.softmax(masked_att_weights, dim=-1)

        # ---- Capture the attention probabilities ----
        captured_attn_probs.append(probs.detach().cpu())

        probs = probs.to(dtype=value_states.dtype)
        att_output = torch.matmul(probs, value_states.permute(0, 2, 1, 3))
        att_output = att_output.permute(0, 2, 1, 3)
        att_output = att_output.reshape(batch_size, -1, num_key_value_heads * num_key_value_groups * head_dim)
        return att_output

    import types
    vwe.eager_attention_forward = types.MethodType(_patched_eager_attention_forward, vwe)
    print("Patched eager_attention_forward to capture attention weights")

    # 2. Discover expected image keys from model config
    from lerobot.configs.types import FeatureType
    model_image_keys = sorted(
        k for k, v in policy.config.input_features.items()
        if v.type == FeatureType.VISUAL
    )
    image_size = policy.config.input_features[model_image_keys[0]].shape[-1] if model_image_keys else 256
    print(f"Model expects {len(model_image_keys)} camera(s): {model_image_keys}")

    # 3. Load offline observation from saved images
    episodes = _find_episode_dirs(SUITE)
    if not episodes:
        raise FileNotFoundError(f"No episode data found for suite '{SUITE}'")

    frames_dir, instruction = episodes[0]
    if not instruction:
        instruction = "unknown task"

    print(f"Suite: {SUITE}")
    print(f"Episode dir: {frames_dir.name}")
    print(f"Task Instruction: {instruction}")
    print(f"Frame index: {FRAME_IDX}")

    # 4. Preprocess for Policy
    batch, img_np = _load_offline_observation(
        frames_dir, FRAME_IDX, instruction, state_dim,
        image_keys=model_image_keys, image_size=image_size, device=device_obj,
    )
    batch = preprocessor(batch)

    # 5. Forward Pass (captures attention via patched method)
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16, enabled=(DEVICE=="cuda")):
        _ = policy.select_action(batch)

    # Restore original method
    vwe.eager_attention_forward = _original_eager_attn

    # 6. Process Attention Weights
    if not captured_attn_probs:
        raise RuntimeError("No attention weights captured.")

    print(f"Captured attention from {len(captured_attn_probs)} forward calls")
    # The first call fills the KV cache (prefix: image + language + state tokens).
    # Subsequent calls are denoising steps (action tokens attending to prefix via KV cache).
    # The first call's attention shows how prefix tokens attend to each other.
    # The denoising calls show how action tokens attend to the prefix.

    # Use the first denoising step's attention (index 1 if available, else 0)
    # In fill_kv_cache call (idx 0): shape is [B, heads, prefix_len, prefix_len]
    # In denoise calls (idx >= 1): shape is [B, heads, action_chunk, prefix_len + action_chunk]
    for i, a in enumerate(captured_attn_probs):
        print(f"  Call {i}: shape {a.shape}")

    if len(captured_attn_probs) > 1:
        # Use first denoising step
        attn = captured_attn_probs[1]  # [B, heads, action_seq, total_seq]
    else:
        attn = captured_attn_probs[0]

    attn = attn.squeeze(0)  # [heads, query_len, kv_len]

    # Average over heads
    attn_avg = attn.mean(dim=0)  # [query_len, kv_len]

    # For denoising steps, query = action tokens, kv = prefix + action tokens
    # Average over query (action) tokens to get attention distribution over kv tokens
    attn_to_prefix = attn_avg.mean(dim=0)  # [kv_len]

    # The prefix contains: [image_tokens..., text_tokens..., state_token]
    # SmolVLA uses 64 visual tokens per image crop per camera
    prefix_len = attn_to_prefix.shape[0]
    visual_len = 64  # tokens for first camera's image
    print(f"KV length: {prefix_len}, using first {visual_len} tokens as visual")

    visual_attention = attn_to_prefix[:visual_len]
    text_attention = attn_to_prefix[visual_len:]

    # 7. Visualization
    img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
    overlay_bgr = overlay_heatmap(img_bgr, visual_attention)
    overlay_rgb = cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Image Heatmap
    axes[0].imshow(overlay_rgb)
    axes[0].set_title(f"Visual Attention (8x8) - {SUITE} frame {FRAME_IDX}")
    axes[0].axis("off")
    
    # Full prefix attention profile
    n_bars = min(40, text_attention.shape[0])
    bar_data = text_attention[:n_bars]
    
    try:
        from transformers import AutoProcessor
        proc = AutoProcessor.from_pretrained(CHECKPOINT)
        # Recreate the prompt to get the exact tokens
        messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": instruction}]}]
        prompt = proc.apply_chat_template(messages, add_generation_prompt=True)
        input_ids = proc.tokenizer(prompt).input_ids
        
        # Decode individual tokens, excluding the <image> placeholder
        all_tokens = [proc.tokenizer.decode([tid]) for tid in input_ids]
        text_tokens = [t for t in all_tokens if "<image>" not in t]
    except Exception as e:
        print(f"Tokenization fallback. Error: {e}")
        text_tokens = instruction.split()

    labels = []
    for i in range(n_bars):
        if i < len(text_tokens):
            # Clean up newlines for display
            disp = text_tokens[i].replace('\n', '\\n')
            labels.append(disp if disp.strip() else "_")
        else:
            labels.append(f"tok_{i}")
            
    axes[1].bar(range(n_bars), bar_data.numpy())
    axes[1].set_xticks(range(n_bars))
    axes[1].set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
    axes[1].set_title("Non-visual Token Attention (action→prefix)")
    axes[1].set_ylabel("Mean Attention Weight")
    
    out_name = f"offline_attention_{SUITE}_frame{FRAME_IDX}.png"
    plt.tight_layout()
    plt.savefig(out_name, dpi=200)
    print(f"Saved visualization to {out_name}")

In [ ]:
if __name__ == "__main__":
    main()